In [1]:
from ent import * 

In [ ]:
# parameters : 

# NOTE : smaller system sizes are used for fast simulation 

graph_qubit = 6
total_qubits = 100 
sparsity = 0.5
avg_gates_per_layer = int((total_qubits//graph_qubit)*sparsity)
layers = 100
num_shots = 100


entropies = []  

graphs = Q6_graph
num_graphs = len(graphs)


for i in trange(num_graphs):

    graph_state = graphs[i]
    entropy  = average_entanglement(graph_state, 
                                    total_qubits, 
                                    avg_gates_per_layer, 
                                    layers, 
                                    num_shots)

    entropies.append(entropy)
    

100%|██████████| 4/4 [00:44<00:00, 11.08s/it]


In [4]:
# saving data - 

for i in range(len(entropies)):
    np.savetxt(f'data/ent-graph{i}.txt', entropies[i])

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))

t = np.arange(0,layers,1)

for i in range(len(entropies)):
    ax.plot(t, entropies[i], label=rf'{i + 1}')

for spine in ax.spines.values():
    spine.set_edgecolor('gray')

plt.tick_params(top=True, left=True, right=True, bottom=True,
               direction="in", axis='both', which='both', 
               labelsize=15, color='k')

font2 = {'family':'serif', 'color':'black', 'size':18}
plt.xlabel(r"$t$", fontdict=font2)
plt.ylabel(r"$\langle S(t)\rangle $", fontdict=font2)

plt.legend(
    fontsize=8,  # Slightly smaller font for external legend
    ncol=1,  # Fewer columns for vertical space
    loc='best', 
    frameon=False,
    borderaxespad=0.  # Removes padding between axes and legend
)

fig.set_dpi(120)
plt.show()

In [5]:
# Finding entanglement velocity =======

def power_law(x, a):
    return a * (x)

power = []; growth_rate = []

i = 0
for ent in entropies:
    x = np.arange(1, layers+1, 1)
    y = ent
    params, covariance = curve_fit(power_law, x, y)
    a_fit = params
    growth_rate.append(a_fit)
    i += 1

In [ ]:
np.savetxt('data/ent_vel-g6.txt',growth_rate)

In [ ]:
# Pair each growth rate with its corresponding group label
groups_and_rates = list(zip([f'G{i+1}' for i in range(len(growth_rate))], graphs, growth_rate))

# Sort by growth rate (ascending order)
sorted_groups = sorted(groups_and_rates, key=lambda x: x[2])

# Print sorted results
for group, gi, rate in sorted_groups:
    print(f'{group} {np.round(rate,4)}')
    # gi.draw_graph()

In [ ]:
# finding the height function 

from graphs import * 


lattice = [i for i in range(graph_qubit)]

graph_ent = []

for Gi in graphs:

    ent = 0
    for i in range(1, graph_qubit):

        partA = lattice[:i]
        partB = lattice[i:]

        circuit = stim.Circuit()
        circuit.append('I', [graph_qubit - 1])

        Gi.apply_to_circuit(circuit, [i for i in range(graph_qubit)], random_order=False)

        simulator = stim.TableauSimulator()
        simulator.do_circuit(circuit)
        tableau = simulator.current_inverse_tableau()
        tab_forward = tableau.inverse()

        ent += (1/(graph_qubit - 1))*ent_state(tab_forward, partA, partB)

    
    graph_ent.append(ent)




# Pair each growth rate with its corresponding group label
groups_and_rates = list(zip([f'G{i+3}' for i in range(graph_qubit)], graphs, graph_ent))

# Sort by growth rate (ascending order)
sorted_groups = sorted(groups_and_rates, key=lambda x: x[2])


# Print sorted results
for group, gi, rate in sorted_groups:
    print(f'{group} {rate}')


G3 1.0
G5 1.0
G4 1.25
G6 1.5
